# Notebook 06: Feature Selection and Importance

## Why This Matters

Adding irrelevant features hurts model performance (especially linear models and KNN), slows training, and makes interpretability harder. In production, every additional feature has a cost: it must be computed, stored, monitored, and maintained. Feature selection is not just a modeling technique - it's an engineering discipline that reduces technical debt. Understanding different methods and when to use each is essential for both competition ML and production ML.

In [ ]:
%pip install -q scikit-learn numpy pandas matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.feature_selection import (SelectKBest, mutual_info_classif, RFE, SelectFromModel)
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import warnings; warnings.filterwarnings("ignore")
np.random.seed(42)
print("Ready.")

## 1. Filter Methods: Fast and Model-Agnostic

Filter methods score each feature independently, without fitting a model.

**Correlation** (Pearson): linear relationship. Fast but misses non-linear dependencies.

**Mutual Information**: measures statistical dependency including non-linear relationships. More powerful than correlation but slower.

**Chi-squared**: for categorical features vs target.

**Variance threshold**: removes near-constant features.

**Limitation**: Filter methods evaluate features independently. A feature useless alone might be valuable in combination. Use filter methods for initial screening only.

In [ ]:
X, y = make_classification(n_samples=2000, n_features=30, n_informative=10, n_redundant=5, n_repeated=5, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
feat_names = [f"f{i}" for i in range(30)]

mi_scores = mutual_info_classif(X_tr, y_tr, random_state=42)
corr_scores = np.abs([np.corrcoef(X_tr[:,i], y_tr)[0,1] for i in range(30)])

scores_df = pd.DataFrame({"feature": feat_names, "mutual_info": mi_scores, "abs_correlation": corr_scores,
                           "is_informative": [i < 10 for i in range(30)]})
print("=== Filter Method Scores (top 15 by MI) ===")
print(scores_df.sort_values("mutual_info", ascending=False).head(15).to_string(index=False))
top10_mi = scores_df.sort_values("mutual_info", ascending=False).head(10)
print(f"\nTop-10 MI features that are truly informative: {top10_mi.is_informative.sum()}/10")

## 2. Wrapper Methods: RFE

Recursive Feature Elimination (RFE) fits the model, ranks features by importance, removes the least important, and repeats. Thorough but O(n_features) model fits.

**When to use**: <1000 features and computational budget allows. RFE is the gold standard when accuracy on a specific model matters most.

In [ ]:
scaler = StandardScaler(); X_tr_s = scaler.fit_transform(X_tr); X_te_s = scaler.transform(X_te)
rfe = RFE(estimator=LogisticRegression(C=0.1, max_iter=500), n_features_to_select=10, step=1)
rfe.fit(X_tr_s, y_tr)
selected = np.array(feat_names)[rfe.support_]
truly_inf = sum(int(f[1:]) < 10 for f in selected)
print(f"RFE selected: {list(selected)}")
print(f"Truly informative in selection: {truly_inf}/10")

lr_all = LogisticRegression(C=0.1, max_iter=500).fit(X_tr_s, y_tr)
lr_rfe = LogisticRegression(C=0.1, max_iter=500).fit(X_tr_s[:,rfe.support_], y_tr)
auc_all = roc_auc_score(y_te, lr_all.predict_proba(X_te_s)[:,1])
auc_rfe = roc_auc_score(y_te, lr_rfe.predict_proba(X_te_s[:,rfe.support_])[:,1])
print(f"\nAUC with all 30 features: {auc_all:.4f}")
print(f"AUC with RFE top-10:      {auc_rfe:.4f}")

## 3. Embedded Methods: Lasso and Tree Importances

**Lasso (L1)**: zeroes out coefficients of unimportant features during training. Very efficient.

**Tree importances (MDI)**: RF and GBM provide feature importance from splitting criterion. Fast but biased toward high-cardinality features.

**Permutation importance**: shuffle each feature and measure performance drop. More accurate than MDI.

In [ ]:
lasso = Lasso(alpha=0.01, max_iter=5000).fit(X_tr_s, y_tr)
lasso_sel = np.array(feat_names)[lasso.coef_ != 0]
print(f"Lasso selected: {len(lasso_sel)}/30 features")
print(f"  Truly informative: {sum(int(f[1:]) < 10 for f in lasso_sel)}/10")

rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)
sfm = SelectFromModel(rf, threshold="mean").fit(X_tr, y_tr)
rf_sel = np.array(feat_names)[sfm.get_support()]
print(f"\nRF SelectFromModel: {len(rf_sel)}/30 features")
print(f"  Truly informative: {sum(int(f[1:]) < 10 for f in rf_sel)}/10")

perm = permutation_importance(rf, X_te, y_te, n_repeats=5, random_state=42)
top10_perm = np.argsort(perm.importances_mean)[-10:][::-1]
print(f"\nPermutation top-10: {[feat_names[i] for i in top10_perm]}")
print(f"  Truly informative: {sum(i < 10 for i in top10_perm)}/10")

## 4. Variance Inflation Factor: Detecting Multicollinearity

VIF measures how much the variance of a coefficient is inflated due to correlation with other features:
`VIF_j = 1 / (1 - R^2_j)` where R^2_j is R^2 from regressing feature j on all others.

- VIF = 1: no correlation
- VIF 5-10: moderate correlation
- VIF > 10: severe multicollinearity - drop feature or use Ridge

In [ ]:
try:
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    X_vif, _ = make_classification(n_samples=500, n_features=10, n_informative=5, n_redundant=3, random_state=42)
    vif_data = pd.DataFrame()
    vif_data["feature"] = [f"f{i}" for i in range(X_vif.shape[1])]
    vif_data["VIF"] = [variance_inflation_factor(X_vif, i) for i in range(X_vif.shape[1])]
    print("=== Variance Inflation Factor ===")
    print(vif_data.sort_values("VIF", ascending=False).to_string(index=False))
    print("\nVIF > 10: severe - drop or use Ridge | VIF 5-10: moderate | VIF < 5: acceptable")
except ImportError:
    print("statsmodels not available. Formula: VIF_j = 1 / (1 - R^2_j)")
    print("R^2_j is from regressing feature j on all other features.")

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Mutual information | Non-linear filter; captures dependencies Pearson misses |
| RFE | Iterative wrapper; most accurate but O(n_features) model fits |
| Lasso selection | Embedded; zeros out unimportant features during training |
| Permutation importance | Model-agnostic; unbiased; measures actual impact on performance |
| VIF | Detects multicollinearity; VIF > 10 = drop or regularize |

### Common Pitfalls
- Running feature selection on the full dataset (leaks test set into feature scores)
- Using MDI importance with high-cardinality features
- Removing highly correlated features without considering which is more reliable
- Using only one method - take intersection across methods
